# 03 — Test des Agents & Tools (P2)

Ce notebook teste :
1. Chaque **tool en isolation** (météo, recherche web, calculatrice)
2. L'**agent LangGraph** complet qui choisit automatiquement le bon outil

**Prérequis** : avoir complété l'installation et configuré `.env` avec `GROQ_API_KEY`.

In [ ]:
import sys
sys.path.insert(0, '..')  # remonter à la racine du projet

from dotenv import load_dotenv
load_dotenv()
print('✅ Environnement chargé')

---
## 1. Test isolé — `get_weather` (OpenMeteo)

In [ ]:
from src.agents.tools import get_weather

# Test 1 : ville française
result = get_weather.invoke({'city': 'Paris'})
print(result)

In [ ]:
# Test 2 : ville internationale
result = get_weather.invoke({'city': 'Dakar'})
print(result)

In [ ]:
# Test 3 : ville introuvable (cas d'erreur)
result = get_weather.invoke({'city': 'VilleQuiNExistePas123'})
print(result)

---
## 2. Test isolé — `web_search` (DuckDuckGo)

In [ ]:
from src.agents.tools import web_search

# Test 1 : actualité climatique récente
result = web_search.invoke({'query': 'catastrophes climatiques 2024 bilan', 'max_results': 3})
print(result)

In [ ]:
# Test 2 : information spécifique
result = web_search.invoke({'query': 'GIEC rapport AR6 conclusions principales', 'max_results': 3})
print(result)

---
## 3. Test isolé — `calculator`

In [ ]:
from src.agents.tools import calculator

tests_calcul = [
    '2 + 2',
    '15 * 1.08',                    # TVA 8%
    'sqrt(144)',                    # racine carrée
    '1000 * (1 + 0.02) ** 10',      # intérêts composés (projection CO2)
    'log10(1000000)',               # logarithme base 10
    '(45.5 - 32) * 5/9',           # Fahrenheit → Celsius
    '1 / 0',                       # division par zéro (cas d'erreur)
]

for expr in tests_calcul:
    result = calculator.invoke({'expression': expr})
    print(result)

---
## 4. Test de l'agent LangGraph complet

L'agent choisit **automatiquement** le bon outil selon la question.

In [ ]:
from src.agents.agent import run_agent

# Question → doit utiliser get_weather
q1 = 'Quelle est la météo actuelle à Lyon ?'
print(f'Question : {q1}')
print(run_agent(q1))
print('─' * 60)

In [ ]:
# Question → doit utiliser calculator
q2 = 'Si les émissions de CO2 augmentent de 2% par an à partir de 37 Gt, '\
     'quel sera le niveau dans 20 ans ?'
print(f'Question : {q2}')
print(run_agent(q2))
print('─' * 60)

In [ ]:
# Question → doit utiliser web_search
q3 = 'Quelles sont les dernières actualités sur les inondations en Europe en 2024 ?'
print(f'Question : {q3}')
print(run_agent(q3))
print('─' * 60)

In [ ]:
# Question → réponse directe sans outil
q4 = 'Bonjour, comment tu t\'appelles ?'
print(f'Question : {q4}')
print(run_agent(q4))
print('─' * 60)

In [ ]:
# Question → combinaison météo + contexte climatique
q5 = 'Il fait combien de degrés à Marseille en ce moment ? '\
     'Est-ce normal pour la saison compte tenu du changement climatique ?'
print(f'Question : {q5}')
print(run_agent(q5))
print('─' * 60)

---
## 5. Vérification de la structure du graphe LangGraph

In [ ]:
from src.agents.agent import build_agent_graph

graph = build_agent_graph()
print('Nœuds du graphe :', list(graph.nodes.keys()))
print('✅ Graphe compilé avec succès')

In [ ]:
# Afficher le schéma ASCII du graphe
graph.get_graph().print_ascii()